# 🔧 Setup and Configuration
## Verify environment, dependencies, and models before running experiments

This notebook performs comprehensive system checks to ensure everything is properly configured for the ColBERT vs Dense Retrieval demo.

In [25]:
# Import shared configuration using setup system
import sys
sys.path.append('../..')  # Add project root to path
from setup import *

print("✅ Setup imported successfully!")
print(f"📂 Working directory: {os.getcwd()}")
print(f"🎯 Project root: {os.getenv('PROJECT_ROOT', 'Not set')}")
print(f"📊 Data directory: {os.getenv('DATA_DIR', 'Not set')}")
vector_store_dir = os.getenv('VECTOR_STORE_DIR', 'vector_store')
print(f"🗄️  Vector store directory: {vector_store_dir}")
print(f"🔧 Device: {get_device()}")

✅ Setup imported successfully!
📂 Working directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
🎯 Project root: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
📊 Data directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data
🗄️  Vector store directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/vector_store
🔧 Device: mps


In [11]:
# 1. Environment Verification
print("🔍 ENVIRONMENT CHECK")
print("=" * 50)

env_vars = {
    'PROJECT_ROOT': 'Project root directory',
    'DATA_DIR': 'Data folder path', 
    'RESTAURANT_REVIEWS_CSV': 'Restaurant reviews file',
    'LANCEDB_PATH': 'Vector database path',
    'DENSE_MODEL_NAME': 'Dense model name',
    'COLBERT_MODEL_NAME': 'ColBERT model name',
    'EMBEDDING_DIMENSION': 'Embedding dimensions',
    'TOP_K_RESULTS': 'Top-K results for search'
}

all_good = True
for var, description in env_vars.items():
    value = os.getenv(var)
    if value:
        print(f"✅ {var:20} : {value}")
    else:
        print(f"❌ {var:20} : NOT SET")
        all_good = False

if all_good:
    print("\n🎉 All environment variables properly configured!")
else:
    print("\n⚠️  Some environment variables missing - check .env file")

🔍 ENVIRONMENT CHECK
✅ PROJECT_ROOT         : /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
✅ DATA_DIR             : /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data
✅ RESTAURANT_REVIEWS_CSV : /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data/restaurant_reviews.csv
❌ LANCEDB_PATH         : NOT SET
✅ DENSE_MODEL_NAME     : all-MiniLM-L6-v2
✅ COLBERT_MODEL_NAME   : sentence-transformers/all-MiniLM-L6-v2
✅ EMBEDDING_DIMENSION  : 384
✅ TOP_K_RESULTS        : 3

⚠️  Some environment variables missing - check .env file


In [12]:
# 2. Dependency Check
print("\n📦 DEPENDENCY CHECK")
print("=" * 50)

dependencies = {
    'torch': 'PyTorch',
    'transformers': 'Hugging Face Transformers',
    'sentence_transformers': 'Sentence Transformers',
    'pylate': 'PyLate (ColBERT)',
    'lancedb': 'LanceDB Vector Database',
    'pandas': 'Pandas Data Analysis',
    'numpy': 'NumPy Scientific Computing',
    'matplotlib': 'Matplotlib Plotting',
    'seaborn': 'Seaborn Statistical Visualization'
}

import importlib
missing_deps = []

for module, name in dependencies.items():
    try:
        imported = importlib.import_module(module)
        version = getattr(imported, '__version__', 'unknown')
        print(f"✅ {name:30} : {version}")
    except ImportError:
        print(f"❌ {name:30} : NOT INSTALLED")
        missing_deps.append(module)

if missing_deps:
    print(f"\n⚠️  Missing dependencies: {', '.join(missing_deps)}")
    print(f"   Install with: pip install {' '.join(missing_deps)}")
else:
    print("\n🎉 All required dependencies installed!")


📦 DEPENDENCY CHECK
✅ PyTorch                        : 2.2.2
✅ Hugging Face Transformers      : 4.55.2
✅ Sentence Transformers          : 3.0.1
✅ PyLate (ColBERT)               : unknown
✅ LanceDB Vector Database        : 0.24.3
✅ Pandas Data Analysis           : 2.3.1
✅ NumPy Scientific Computing     : 1.26.4
✅ Matplotlib Plotting            : 3.10.5
✅ Seaborn Statistical Visualization : 0.13.2

🎉 All required dependencies installed!


In [13]:
# 3. Device Configuration Check
print("\n🔧 DEVICE CONFIGURATION")
print("=" * 50)

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"MPS (M1/M2) available: {torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

selected_device = get_device()
print(f"Selected device: {selected_device}")

if selected_device == 'mps':
    print("🍎 M1/M2 GPU detected - optimizing for Apple Silicon")
    print("   Note: Some models work better on CPU for M1 Macs")
elif selected_device == 'cuda':
    print(f"🔥 CUDA GPU detected")
    print(f"   GPU name: {torch.cuda.get_device_name()}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("💻 Using CPU - should work fine for demo dataset")

# Memory check
if torch.backends.mps.is_available():
    print(f"MPS built: {torch.backends.mps.is_built()}")
    
print(f"\n✅ Device configuration: {selected_device}")


🔧 DEVICE CONFIGURATION
PyTorch version: 2.2.2
MPS (M1/M2) available: True
CUDA available: False
Selected device: mps
🍎 M1/M2 GPU detected - optimizing for Apple Silicon
   Note: Some models work better on CPU for M1 Macs
MPS built: True

✅ Device configuration: mps


In [14]:
# 4. Model Loading Test
print("\n🤖 MODEL LOADING TEST")
print("=" * 50)

# Test Dense Model Loading
print("\n📦 Testing Dense Model...")
try:
    from sentence_transformers import SentenceTransformer
    
    # Use CPU for M1 compatibility
    device_for_dense = 'cpu' if get_device() == 'mps' else get_device()
    print(f"   Loading on device: {device_for_dense}")
    
    dense_model = SentenceTransformer(os.getenv('DENSE_MODEL_NAME'), device=device_for_dense)
    
    # Quick test
    test_text = "test embedding"
    test_embedding = dense_model.encode(test_text)
    
    print(f"✅ Dense model loaded successfully")
    print(f"   Model: {os.getenv('DENSE_MODEL_NAME')}")
    print(f"   Device: {device_for_dense}")
    print(f"   Embedding shape: {test_embedding.shape}")
    
    # Free memory
    del dense_model, test_embedding
    
except Exception as e:
    print(f"❌ Dense model failed: {e}")

# Test ColBERT Model Loading
print("\n🔍 Testing ColBERT Model...")
try:
    from pylate import models
    
    # ColBERT works better on CPU for M1
    device_for_colbert = 'cpu'
    print(f"   Loading on device: {device_for_colbert}")
    
    colbert_model = models.ColBERT(
        model_name_or_path=os.getenv('COLBERT_MODEL_NAME'),
        device=device_for_colbert
    )
    
    # Quick test
    test_embedding = colbert_model.encode([test_text], is_query=False)
    
    print(f"✅ ColBERT model loaded successfully")
    print(f"   Model: {os.getenv('COLBERT_MODEL_NAME')}")
    print(f"   Device: {device_for_colbert}")
    print(f"   Embedding shape: {test_embedding[0].shape} (tokens x dimensions)")
    
    # Free memory
    del colbert_model, test_embedding
    
except Exception as e:
    print(f"❌ ColBERT model failed: {e}")

print("\n🎉 Model loading tests complete!")


🤖 MODEL LOADING TEST

📦 Testing Dense Model...
   Loading on device: cpu
✅ Dense model loaded successfully
   Model: all-MiniLM-L6-v2
   Device: cpu
   Embedding shape: (384,)

🔍 Testing ColBERT Model...
   Loading on device: cpu


The checkpoint does not contain a linear projection layer. Adding one with output dimensions (384, 128).
Created a PyLate model from base encoder.


✅ ColBERT model loaded successfully
   Model: sentence-transformers/all-MiniLM-L6-v2
   Device: cpu
   Embedding shape: (7, 128) (tokens x dimensions)

🎉 Model loading tests complete!


In [15]:
# 5. Data Verification
print("\n📊 DATA VERIFICATION")
print("=" * 50)

# Check if data file exists
data_path = os.getenv('RESTAURANT_REVIEWS_CSV')
print(f"Data file path: {data_path}")

if os.path.exists(data_path):
    try:
        df = pd.read_csv(data_path)
        
        print(f"✅ CSV file loaded successfully")
        print(f"   Rows: {len(df):,}")
        print(f"   Columns: {list(df.columns)}")
        
        # Validate required columns
        required_cols = ['id', 'restaurant', 'review', 'reviewer', 'rating']
        missing_cols = [col for col in required_cols if col not in df.columns]
        
        if missing_cols:
            print(f"❌ Missing required columns: {missing_cols}")
        else:
            print(f"✅ All required columns present: {required_cols}")
            
        # Data quality checks
        print(f"\nData Quality:")
        print(f"   Null values: {df.isnull().sum().sum()}")
        print(f"   Rating range: {df['rating'].min()} - {df['rating'].max()}")
        print(f"   Average rating: {df['rating'].mean():.1f}")
        print(f"   Unique restaurants: {df['restaurant'].nunique()}")
        
        # Sample data
        print(f"\n📝 Sample review:")
        sample = df.iloc[0]
        print(f"   Restaurant: {sample['restaurant']}")
        print(f"   Rating: {'⭐' * sample['rating']}")
        print(f"   Review: {sample['review'][:100]}...")
        
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
else:
    print(f"❌ Data file not found: {data_path}")
    print("   Please check the path in .env file")


📊 DATA VERIFICATION
Data file path: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data/restaurant_reviews.csv
✅ CSV file loaded successfully
   Rows: 12
   Columns: ['id', 'restaurant', 'review', 'reviewer', 'rating']
✅ All required columns present: ['id', 'restaurant', 'review', 'reviewer', 'rating']

Data Quality:
   Null values: 0
   Rating range: 1 - 5
   Average rating: 3.8
   Unique restaurants: 12

📝 Sample review:
   Restaurant: Mario's Bistro
   Rating: ⭐⭐⭐⭐⭐
   Review: OMG this little Italian place is a hidden gem! 😍 Went there last night with my boyfriend and we sat ...


In [18]:
print(os.getenv('VECTOR_STORE_DIR', ))

/Users/luvsuneja/Documents/repos/advanced-rag-experimentation/vector_store


In [23]:
# 6. LanceDB Vector Database Test
print("\n🗄️ VECTOR DATABASE TEST")
print("=" * 50)

# Test LanceDB connection
db_path = os.getenv('VECTOR_STORE_DIR',)
print(f"Database path: {db_path}")

try:
    import lancedb
    
    # Connect to database
    db = lancedb.connect(db_path)
    print(f"✅ LanceDB connected successfully")
    
    # Check existing tables
    tables = db.table_names()
    if tables:
        print(f"   Existing tables: {tables}")
        for table_name in tables:
            table = db.open_table(table_name)
            print(f"     📊 {table_name}: {len(table)} records")
    else:
        print(f"   📭 No existing tables (will be created during embedding process)")
    
    # Test table creation (cleanup after)
    try:
        import pyarrow as pa
        test_data = [{"id": 1, "text": "test", "vector": [0.1, 0.2, 0.3]}]
        test_schema = pa.schema([
            pa.field("id", pa.int64()),
            pa.field("text", pa.string()),
            pa.field("vector", pa.list_(pa.float32()))
        ])
        
        # Create and immediately drop test table
        test_table = db.create_table("test_table", test_data, schema=test_schema)
        db.drop_table("test_table")
        print(f"✅ Table operations working correctly")
        
    except Exception as e:
        print(f"⚠️  Table operation test failed: {e}")
    
except Exception as e:
    print(f"❌ LanceDB connection failed: {e}")
    print("   This might be okay - database will be created when needed")


🗄️ VECTOR DATABASE TEST
Database path: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/vector_store
✅ LanceDB connected successfully
   📭 No existing tables (will be created during embedding process)
✅ Table operations working correctly


In [24]:
# 7. Configuration Summary
print("\n" + "="*60)
print("📋 CONFIGURATION SUMMARY")
print("="*60)

# Display final configuration for confirmation
config_summary = {
    "🔧 Device": get_device(),
    "📦 Dense Model": os.getenv('DENSE_MODEL_NAME'),
    "🔍 ColBERT Model": os.getenv('COLBERT_MODEL_NAME'),
    "📊 Data Path": os.getenv('RESTAURANT_REVIEWS_CSV'),
    "🗄️ Vector DB": os.getenv('LANCEDB_PATH'),
    "📏 Embedding Dim": os.getenv('EMBEDDING_DIMENSION', '384'),
    "🔢 Top K Results": os.getenv('TOP_K_RESULTS', '3'),
    "🎯 Max Reviews": os.getenv('MAX_REVIEWS', '12')
}

for key, value in config_summary.items():
    print(f"{key:18} : {value}")

# Performance expectations
print(f"\n🚀 PERFORMANCE EXPECTATIONS:")
selected_device = get_device()
if selected_device == 'mps':
    print("   ⚡ M1/M2 GPU will accelerate some operations")
    print("   📝 Using CPU for model compatibility")
    print("   ⏱️  Expected embedding time: ~30 seconds for 12 reviews")
elif selected_device == 'cuda':
    print("   🔥 CUDA GPU will significantly accelerate operations")
    print("   ⏱️  Expected embedding time: ~10 seconds for 12 reviews")
else:
    print("   💻 CPU processing will be slower but reliable")
    print("   ⏱️  Expected embedding time: ~1-2 minutes for 12 reviews")

print(f"\n✅ SETUP COMPLETE! Ready for embeddings creation.")
print(f"🎯 Next: Run notebook 2-Dense-Embeddings.ipynb")

# Optional: Show system memory info
import psutil
memory = psutil.virtual_memory()
print(f"\n💾 System Memory: {memory.available / (1024**3):.1f}GB available of {memory.total / (1024**3):.1f}GB total")


📋 CONFIGURATION SUMMARY
🔧 Device           : mps
📦 Dense Model      : all-MiniLM-L6-v2
🔍 ColBERT Model    : sentence-transformers/all-MiniLM-L6-v2
📊 Data Path        : /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data/restaurant_reviews.csv
🗄️ Vector DB       : None
📏 Embedding Dim    : 384
🔢 Top K Results    : 3
🎯 Max Reviews      : 12

🚀 PERFORMANCE EXPECTATIONS:
   ⚡ M1/M2 GPU will accelerate some operations
   📝 Using CPU for model compatibility
   ⏱️  Expected embedding time: ~30 seconds for 12 reviews

✅ SETUP COMPLETE! Ready for embeddings creation.
🎯 Next: Run notebook 2-Dense-Embeddings.ipynb

💾 System Memory: 1.0GB available of 16.0GB total
